# Project - Evaluate Models on FPN Search using Azure API
Searching for a good cheap model for FPNsearch Note the Azure data is not in RAG form

Resources:
* [Azure AI Search client library for Python - version 11.6.0](https://learn.microsoft.com/en-us/python/api/overview/azure/search-documents-readme?view=azure-python)

In [24]:
import os
import json
import random

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

from datetime import datetime
import re
from huggingface_hub import HfApi, CommitOperationAdd


import subprocess
from IPython.display import Markdown, display

import asyncio
from bd_api import BdApiClient, AsyncBdApiClient
from bd_api.errors import BdApiRequestError, BdApiStreamError

In [25]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:4]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


# for RAG
# Fetch the API endpoint (supports both http and https)
BD_API_BASE_URL = os.getenv("BD_API_BASE_URL")
if not BD_API_BASE_URL:
    raise ValueError("BD_API_BASE_URL environment variable is missing. Please set it in your .env file.")

# Fetch the API Key
BD_API_KEY = os.getenv("BD_API_KEY")
if not BD_API_KEY:
    print("Warning: BD_API_KEY environment variable is missing. Requests may fail if the server requires an API key.")

print(f"Using API Base URL: {BD_API_BASE_URL}")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
Grok API Key exists and begins xai-
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-o
Using API Base URL: https://ai.jake.atmoapps.net


In [26]:
search_client = None
openai = OpenAI(api_key=openai_api_key)


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)










In [27]:
# OPENAI_MODEL = "gpt-5"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
# GEMINI_MODEL = "gemini-2.5-pro"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"





#tried models that did not work well at all:
#"gemma-4-26b-a4b-it": gemini, "gemma-4-31b-it": gemini


#models that work:
#"claude-haiku-4-5": anthropic,
#"gemini-3.1-flash-lite-preview": gemini

#trying to get this model to work - but cannot be found: "gpt-oss-120b": openai,,"gemma-4-31B": gemini,

CLIENTS = {
    "openai/gpt-oss-120b": openrouter,
    "ai21/jamba-large-1.7":openrouter,
    "amazon/nova-pro-v1":openrouter,
    "meta-llama/llama-3.3-70b-instruct":openrouter,
    "minimax/minimax-m2.5":openrouter,
    "moonshotai/kimi-k2-thinking":openrouter
    } 
MODELS= list(CLIENTS.keys())
print(MODELS)

EVAL_CLIENTS = {"gpt-4.1-mini":openai} #,"claude-sonnet-4-5":anthropic
EVAL_MODELS= list(EVAL_CLIENTS.keys())



['openai/gpt-oss-120b', 'ai21/jamba-large-1.7', 'amazon/nova-pro-v1', 'meta-llama/llama-3.3-70b-instruct', 'minimax/minimax-m2.5', 'moonshotai/kimi-k2-thinking']


In [28]:
#MODELS= ['gpt-4.1-mini']

# MODELS = ['gpt-oss-120b','google/gemma-4-31B-it']



In [29]:
def RAG_search(query, nTopResults = 5, nCandidates = 20):
    """
    Perform a RAG search using the BD API.

    Args:
        query (str): The search query.
        nTopResults (int): The number of top results to return.

    Returns:
        list: A list of search results.
    """
    with BdApiClient(base_url=BD_API_BASE_URL, api_key=BD_API_KEY) as client:
        try:
            # Use reranking for better relevance
            retrieve_response = client.retrieve.create(
                question=query,
                retrieval={
                    "retrieval_mode": "rerank",
                    "candidate_count": nCandidates,
                    "result_count": nTopResults
                }
            )

            print("Complete Raw API Response:")
            raw_json = json.dumps(retrieve_response.raw_json, indent=2)
            print(raw_json)

            print("\nNormalized Retrieved Items JSON (first item):")
            if retrieve_response.retrieved_items:
                print(json.dumps(retrieve_response.retrieved_items[0], indent=2))

            print("\nPrepared Prompt Input JSON:")
            prepared_json = json.dumps(retrieve_response.prepared_prompt_input, indent=2)
            print(prepared_json)

            # NEW: Access retrieval metadata
            print("\n--- NEW: Retrieval Metadata ---")
            print(f"Requested mode: {retrieve_response.retrieval.requested_mode}")
            print(f"Effective mode: {retrieve_response.retrieval.effective_mode}")
            print(f"Fallback used: {retrieve_response.retrieval.fallback_used}")
            if retrieve_response.retrieval.fallback_used:
                print(f"Fallback reason: {retrieve_response.retrieval.fallback_reason}")
            print(f"Candidates retrieved: {retrieve_response.retrieval.initial_retrieved_count}")
            print(f"Final results: {retrieve_response.retrieval.final_retrieved_count}")

            # Cost breakdown
            retrieval_cost = retrieve_response.cost.get('retrievalEstimated', {}).get('dollars', 0)
            rerank_cost = retrieve_response.cost.get('rerankingEstimated', {}).get('dollars', 0)
            print(f"\nRetrieval Cost: ${retrieval_cost:.6f}")
            print(f"Reranking Cost: ${rerank_cost:.6f}")


            # Return the results
            return prepared_json

        except BdApiRequestError as e:
            print(f"Retrieve Request failed. HTTP {e.status_code}")
            return []
        except BdApiStreamError as e:
            print(f"Stream error: {e}")
            return []
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return []


In [30]:
with open("system_question.txt", "r") as f:
    system_message = f.read()

with open("system_evaluator.md", "r") as f:
    system_message_evaluator = f.read()


In [31]:
def get_references(json_data,root_path="https://fpnotebook.com/"):
    references = []
    for result in json_data:
        print(result);
        #content = result['metadata']
        #page_url = content['web_url']
        #title = content['monograph_title']
        #references.append(f"[{title}]({page_url}")
    return  "\n\n**References from FPnotebook:**\n" + "\n".join(references)

In [32]:
include_history = False
def chat(message, history, search_result, model, client):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    if not include_history:
        history = []        
    
    message_new = f"""User question: {message}\n\n
    Related Information from FPNotebook in JSON: {search_result}"""
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message_new}]

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [33]:
def evaluate_answer(question, answer, model, client):
    prompt = f"""You are an expert evaluator of question-answering. Please evaluate the quality of the answer based on the question. Here is the question: {question} and here is the answer: {answer}."""
    messages = [{"role": "system", "content": system_message_evaluator}, {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=model, messages=messages, response_format={"type": "json_object"})
    result = response.choices[0].message.content
    eval_json = json.loads(result)
    return eval_json

In [ ]:
questionID = "FPN_004"
question = "List the common contraindications to beta blocker therapy."
search_result = RAG_search(question)
print(f"Search result for question '{question}': {search_result}")

references = get_references(search_result)
print(f"References for question '{question}': {references}")

data = {}
data['question'] = question
data['references'] = references
data['models']={}
for model in MODELS:
    print(f"Testing model: {model}")
    answer = chat(question, [], search_result, model, CLIENTS[model])
    print(f"Answer from {model}: {answer}")
    data['models'][model] = {}
    data['models'][model]['answer'] = answer
    data['models'][model]['evals'] = {}
    for eval_model in EVAL_MODELS:
        eval_json = evaluate_answer(question, answer, eval_model, EVAL_CLIENTS[eval_model])
        print(f"Evaluation of {model} by {eval_model}: {eval_json}")
        data['models'][model]['evals'][eval_model] = eval_json

with open(questionID + '.json', 'w') as outfile:
    json.dump(data, outfile)

Complete Raw API Response:
{
  "knowledgeBaseId": "AV0IEYJYTF",
  "retrieval": {
    "requestedMode": "rerank",
    "effectiveMode": "rerank",
    "fallbackUsed": false,
    "candidateCount": 20,
    "resultCount": 5,
    "rerankModelArn": "arn:aws:bedrock:us-east-1::foundation-model/cohere.rerank-v3-5:0",
    "initialRetrievedCount": 20,
    "finalRetrievedCount": 5,
    "deduplication": {
      "applied": true,
      "itemsBeforeDedup": 20,
      "itemsAfterDedup": 12,
      "duplicatesRemoved": 8
    }
  },
  "rawRetrievalResponse": {
    "retrievalResults": [
      {
        "content": {
          "type": "TEXT",
          "text": "Title: Beta Blocker Path: Cardiovascular Medicine Book > Pharmacology Chapter > Beta Blocker Aliases: Beta-Blocker, Beta Adrenergic Antagonist, Beta Adrenoceptor Blocking Drug, Penbutalol, Levatol, Oxprenolol, Carteolol  History - Sir James Black and Propranolol: Sir James Black won 1988 Nobel Prize for Propranolol; Synthesized Propranolol first in the 1